In [ ]:
# @title Set ONLINE=False if running locally
ONLINE = True

In [ ]:
# @title Imports
import pandas as pd
import numpy as np
import math
import gdown
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
import torch.nn.functional as F

In [ ]:
# Download training and submission data
if ONLINE:
    url = 'https://drive.google.com/file/d/1lDEtOq644qlaw_vh0i5c3ChVfrHJiuMc/view?usp=sharing'
    output = 'data_train.csv'
    gdown.download(url, output, fuzzy=True)
    url = 'https://drive.google.com/file/d/1KMiakfzTWYr4QJU9S5DC9e5KIo_8bb24/view?usp=sharing'
    output = 'sampleSubmission.csv'
    gdown.download(url, output, fuzzy=True)

In [ ]:
# load training data
number_of_users, number_of_movies = (10000, 1000)
data_pd = pd.read_csv('data_train.csv')

# Split the dataset into train and test
train_size = 0.9
train_pd, test_pd = train_test_split(data_pd, train_size=train_size, random_state=42)

In [ ]:
def extract_users_items_predictions(data_pd):
    users, movies = \
        [np.squeeze(arr) for arr in np.split(data_pd.Id.str.extract('r(\d+)_c(\d+)').values.astype(int) - 1, 2, axis=-1)]
    predictions = data_pd.Prediction.values
    return users, movies, predictions

In [ ]:
# extract matrix data
train_users, train_movies, train_predictions = extract_users_items_predictions(train_pd)
test_users, test_movies, test_predictions = extract_users_items_predictions(test_pd)

In [ ]:
# create full matrix of observed values
data = np.full((number_of_users, number_of_movies), 0)
mask = np.full((number_of_users, number_of_movies), 0) # 0 -> unobserved value, 1->observed value

for user, movie, pred in zip(train_users, train_movies, train_predictions):
    data[user][movie] = pred
    mask[user][movie] = 1

In [ ]:
# define RMSE metric
rmse = lambda x, y: math.sqrt(mean_squared_error(x, y))

def get_score(predictions, target_values=test_predictions):
    return rmse(predictions, target_values)

def extract_prediction_from_full_matrix(reconstructed_matrix, users=test_users, movies=test_movies):
    # returns predictions for the users-movies combinations specified based on a full m \times n matrix
    assert(len(users) == len(movies)), "users-movies combinations specified should have equal length"
    predictions = np.zeros(len(users))

    for i, (user, movie) in enumerate(zip(users, movies)):
        predictions[i] = reconstructed_matrix[user][movie]

    return predictions

In [ ]:
# @title Use GPU in colab: Runtime->Change Runtime type

#  use gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using device:', device)

In [ ]:
# @title Model definition
class NCF(nn.Module):
    def __init__(self, n_users, n_items, n_factors,users_embedding_size, movies_embedding_size):
        super(NCF, self).__init__()

        # GMF part
        self.embedding_gmf_P = nn.Embedding(n_users, n_factors)
        self.embedding_gmf_Q = nn.Embedding(n_items, n_factors)

        # MLP part
        self.embedding_mlp_P = nn.Embedding(n_users,users_embedding_size)
        self.embedding_mlp_Q = nn.Embedding(n_items, movies_embedding_size)
        self.mlp = nn.Sequential(
            nn.Linear(in_features=users_embedding_size +
                      movies_embedding_size, out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=n_factors),
            nn.ReLU(),

        )
        #Final layer before scaling
        self.final= nn.Sequential(nn.Linear(in_features=n_factors*2, out_features=1),nn.Sigmoid())

    def forward(self, user_input, item_input):
        # GMF part
        gmf_p = self.embedding_gmf_P(user_input)
        gmf_q = self.embedding_gmf_Q(item_input)
        gmf_vector = gmf_p * gmf_q

        # MLP part
        mlp_p = self.embedding_mlp_P(user_input)
        mlp_q = self.embedding_mlp_Q(item_input)
        mlp_vector = self.mlp(torch.cat([mlp_p, mlp_q], dim=-1))
        # Concatenate GMF and MLP parts
        ncf_vector = torch.cat([gmf_vector, mlp_vector], dim=-1)
        output = self.final(ncf_vector)

        return 1 + 4 * output  # Map output to [1, 5]


In [ ]:
# @title Hyperparameters
batch_size = 128
num_epochs = 30
show_validation_score_every_epochs = 1
n_factors=16
movies_embedding_size= 16
users_embedding_size=32
learning_rate = 1e-4
weight_decay= 1e-4 # 1.1184

In [ ]:
# @title Model instantiation
def mse_loss(predictions, target):
    return torch.mean((predictions - target) ** 2)

ncf = NCF(number_of_users, number_of_movies, n_factors,users_embedding_size,movies_embedding_size).to(device)

optimizer = optim.AdamW(ncf.parameters())
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)
#scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.01, steps_per_epoch=len(train_dataloader), epochs=num_epochs)


In [ ]:
# @title Dataloaders
train_users_torch = torch.tensor(train_users, device=device)
train_movies_torch = torch.tensor(train_movies, device=device)
train_predictions_torch = torch.tensor(train_predictions, device=device)

train_dataloader = DataLoader(
    TensorDataset(train_users_torch, train_movies_torch, train_predictions_torch),
    batch_size=batch_size)

test_users_torch = torch.tensor(test_users, device=device)
test_movies_torch = torch.tensor(test_movies, device=device)

test_dataloader = DataLoader(
    TensorDataset(test_users_torch, test_movies_torch),
    batch_size=batch_size)

In [ ]:
# @title Training

step = 0
with tqdm(total=len(train_dataloader) * num_epochs) as pbar:
    for epoch in range(num_epochs):
        for users_batch, movies_batch, target_predictions_batch in train_dataloader:
            optimizer.zero_grad()

            predictions_batch = ncf(users_batch, movies_batch)

            loss = mse_loss(predictions_batch, target_predictions_batch)

            loss.backward()

            optimizer.step()
            pbar.update(1)
            step += 1

        if epoch % show_validation_score_every_epochs == 0:

            with torch.no_grad():
                all_predictions = []
                for users_batch, movies_batch in test_dataloader:
                    predictions_batch = ncf(users_batch, movies_batch)
                    all_predictions.append(predictions_batch)

            all_predictions = torch.cat(all_predictions)

            reconstuction_rmse = get_score(all_predictions.cpu().numpy())
            scheduler.step(reconstuction_rmse)
            pbar.set_description('At epoch {:3d} loss is {:.4f}'.format(epoch, reconstuction_rmse))
            print('training/test loss --->', np.sqrt(loss.item()),"/",reconstuction_rmse, "learning rate--> ",scheduler.get_last_lr()[0])
